# Secondary SOC+NMR model coefficients

Coefficients at `lambda.min` from the SOC+NMR models saved with `fit_full = TRUE` in the diabetes, kidney, and MASLD complication notebooks. Clinical covariates are listed first, then Nightingale biomarkers in `rowdata` order. A blank cell means that covariate was not in the model.


In [ ]:
library(glmnet)
library(tidyverse)
library(here)


In [ ]:
feature_order <- c(
  "age_at_sample",
  "sex",
  "time_with_diab",
  "time_with_ckd",
  "time_with_nafld",
  "BMI_interp",
  "SBP_interp",
  "M_a1c",
  "A1C_obs",
  "T_a1c",
  "M_a1c_var",
  "A1C_cv_obs",
  "T_a1c_var",
  "M_egfr",
  "EGFR_obs",
  "T_egfr",
  "M_egfr_slope",
  "EGFR_slope_obs",
  "T_egfr_slope",
  "M_nfs",
  "NFS_obs",
  "T_nfs",
  "M_ldl",
  "LDL_obs",
  "T_ldl",
  "M_hdl_choles",
  "HDL_CHOLES_obs",
  "T_hdl_choles",
  "M_triglycerides",
  "TRIGLYCERIDES_obs",
  "T_triglycerides",
  "M_albumin",
  "ALBUMIN_obs",
  "T_albumin",
  "M_hgb",
  "HGB_obs",
  "T_hgb",
  "M_smoke",
  "smk_current",
  "smk_former",
  "diabetes_at_baseline",
  "bp_treatment",
  "statin_treatment"
)

feature_labels <- c(
  age_at_sample        = "Age at enrollment",
  sex                  = "Sex (Male = 1, Female = 0)",
  time_with_diab       = "Time with diabetes at enrollment",
  time_with_ckd        = "Time with CKD at enrollment",
  time_with_nafld      = "Time with MASLD at enrollment",
  BMI_interp           = "Body-Mass-Index",
  SBP_interp           = "Systolic Blood Pressure",
  M_a1c                = "Missingness HbA1c indicator",
  A1C_obs              = "Observed HbA1c value",
  T_a1c                = "Time delay HbA1c measurement",
  M_a1c_var            = "Missingness HbA1c variability indicator",
  A1C_cv_obs           = "Observed HbA1c coefficient of variation",
  T_a1c_var            = "Time delay HbA1c variability measurement",
  M_egfr               = "Missingness eGFR indicator",
  EGFR_obs             = "Observed eGFR value",
  T_egfr               = "Time delay eGFR measurement",
  M_egfr_slope         = "Missingness eGFR slope indicator",
  EGFR_slope_obs       = "Observed eGFR slope",
  T_egfr_slope         = "Time delay eGFR slope measurement",
  M_nfs                = "Missingness NFS indicator",
  NFS_obs              = "Observed NAFLD fibrosis score",
  T_nfs                = "Time delay NFS measurement",
  M_ldl                = "Missingness LDL-C indicator",
  LDL_obs              = "Observed LDL-C value",
  T_ldl                = "Time delay LDL-C measurement",
  M_hdl_choles         = "Missingness HDL-C indicator",
  HDL_CHOLES_obs       = "Observed HDL-C value",
  T_hdl_choles         = "Time delay HDL-C measurement",
  M_triglycerides      = "Missingness triglycerides indicator",
  TRIGLYCERIDES_obs    = "Observed triglycerides value",
  T_triglycerides      = "Time delay triglycerides measurement",
  M_albumin            = "Missingness albumin indicator",
  ALBUMIN_obs          = "Observed albumin value",
  T_albumin            = "Time delay albumin measurement",
  M_hgb                = "Missingness hemoglobin indicator",
  HGB_obs              = "Observed hemoglobin value",
  T_hgb                = "Time delay hemoglobin measurement",
  M_smoke              = "Missingness Smoking indicator",
  smk_current          = "Smoking reported at enrollment",
  smk_former           = "Former Smoking reported at enrollment",
  diabetes_at_baseline = "Prior diabetes dx at enrollment",
  bp_treatment         = "Blood pressure rx at time of enrollment",
  statin_treatment     = "Statin rx at time of enrollment"
)


In [ ]:
diabetes_dir <- here("outputs/secondary_diabetes")
kidney_dir <- here("outputs/secondary_kidney")
liver_dir <- here("outputs/secondary_liver")

# SOC+NMR fits saved with fit_full = TRUE
model_files <- tibble::tribble(
  ~label, ~path,
  "T2D | Chronic Kidney Disease",   file.path(diabetes_dir, "t2d_ckd_marker_nmr_oof.rds"),
  "T2D | Broad Renal Disease",      file.path(diabetes_dir, "t2d_nephropathy_marker_nmr_oof.rds"),
  "T2D | Heart Failure",            file.path(diabetes_dir, "t2d_heart_failure_marker_nmr_oof.rds"),
  "T2D | Neuropathy",               file.path(diabetes_dir, "t2d_neuropathy_marker_nmr_oof.rds"),
  "T2D | Foot Pathology",           file.path(diabetes_dir, "t2d_foot_marker_nmr_oof.rds"),
  "T2D | Retinopathy",              file.path(diabetes_dir, "t2d_retinopathy_marker_nmr_oof.rds"),
  "CKD | Heart Failure",            file.path(kidney_dir, "ckd_heart_failure_marker_nmr_oof.rds"),
  "CKD | Coronary Heart Disease",   file.path(kidney_dir, "ckd_chd_marker_nmr_oof.rds"),
  "CKD | Atrial Fibrillation",      file.path(kidney_dir, "ckd_afib_marker_nmr_oof.rds"),
  "CKD | Kidney Failure",           file.path(kidney_dir, "ckd_esrd_marker_nmr_oof.rds"),
  "CKD | Acute MI/CS",              file.path(kidney_dir, "ckd_acs_marker_nmr_oof.rds"),
  "CKD | TIA/Stroke",               file.path(kidney_dir, "ckd_tia_stroke_marker_nmr_oof.rds"),
  "MASLD | Type 2 Diabetes",        file.path(liver_dir, "nafld_diab_marker_nmr_oof.rds"),
  "MASLD | Kidney Disease",         file.path(liver_dir, "nafld_ckd_marker_nmr_oof.rds"),
  "MASLD | Portal Hypertension",    file.path(liver_dir, "nafld_portalht_marker_nmr_oof.rds"),
  "MASLD | Venous Thrombosis",      file.path(liver_dir, "nafld_venous_thrombosis_marker_nmr_oof.rds")
)

missing <- model_files$path[!file.exists(model_files$path)]
if (length(missing) > 0) {
  stop("Missing SOC+NMR RDS file(s):\n", paste(missing, collapse = "\n"))
}

extract_glmnet_coef <- function(model) {
  coef(model$full_model$model$glmnet.fit, s = model$full_model$model$lambda.min) %>%
    as.matrix() %>%
    as.data.frame() %>%
    tibble::rownames_to_column("feature") %>%
    dplyr::rename(coef = 2) %>%
    filter(feature != "(Intercept)")
}

coef_list <- lapply(seq_len(nrow(model_files)), function(i) {
  label <- model_files$label[[i]]
  extract_glmnet_coef(readRDS(model_files$path[[i]])) %>%
    select(feature, !!label := coef)
})

joined <- purrr::reduce(coef_list, full_join, by = "feature")

rowdata <- read.csv(here("data/rowdata.csv")) %>%
  distinct(name, Excel_column_name)

nmr_features <- intersect(rowdata$name, joined$feature)
soc_features <- setdiff(joined$feature, nmr_features)
row_order <- c(
  intersect(feature_order, soc_features),
  setdiff(soc_features, feature_order),
  nmr_features
)

joined <- joined %>%
  mutate(feature = factor(feature, levels = row_order)) %>%
  arrange(feature) %>%
  mutate(feature = as.character(feature)) %>%
  left_join(rowdata, by = c("feature" = "name")) %>%
  mutate(
    `Feature Name` = coalesce(
      unname(feature_labels[feature]),
      Excel_column_name,
      feature
    )
  ) %>%
  select(
    Feature = feature,
    `Feature Name`,
    all_of(model_files$label)
  )

joined %>% head()

out_dir <- here("outputs/tables")
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
write.csv(joined, file.path(out_dir, "SECONDARY_COEFF.csv"), row.names = FALSE)

# Consolidated Event Counts


In [ ]:
# consolidate C-index stats and event counts - taken from each oof fit object
fmt_cindex <- function(res) {
  sprintf("%.3f (%.3f, %.3f)", res$c_index, res$ci_lower, res$ci_upper)
}

spec_files <- c(
  "BASE"         = "base",
  "BASE+NMR"     = "base_nmr",
  "CLIN"         = "clin",
  "CLIN+NMR"     = "clin_nmr",
  "SOC+"         = "marker",
  "SOC(+) +NMR"  = "marker_nmr"
)

secondary_tte <- dplyr::bind_rows(lapply(seq_len(nrow(model_files)), function(i) {
  prefix <- sub("_marker_nmr_oof\\.rds$", "", model_files$path[[i]])
  parts <- strsplit(model_files$label[[i]], " \\| ")[[1]]

  fits <- lapply(spec_files, function(spec) {
    path <- paste0(prefix, "_", spec, "_oof.rds")
    if (!file.exists(path)) stop("Missing OOF RDS: ", path)
    readRDS(path)
  })

  n <- vapply(fits, function(res) res$n, numeric(1))
  n_events <- vapply(fits, function(res) res$n_events, numeric(1))
  if (length(unique(n)) != 1L || length(unique(n_events)) != 1L) {
    stop("Event counts differ across specs for ", model_files$label[[i]])
  }

  tibble::tibble(
    `Prevalent Condition` = parts[[1]],
    `Events (n)` = as.integer(n_events[[1]]),
    `Censored (n)` = as.integer(n[[1]] - n_events[[1]]),
    Endpoint = parts[[2]],
    !!!lapply(fits, fmt_cindex)
  )
}))

secondary_tte

write.csv(
  secondary_tte,
  here("outputs/tables", "CINDEX_STATS_SECONDARY.csv"),
  row.names = FALSE
)
